# 07 — Window Functions

**What's in this notebook:**
- The `OVER` clause — aggregate-like computations that **keep the row** instead of collapsing it
- `PARTITION BY` — splitting rows into independent windows (like `GROUP BY` but non-destructive)
- `ORDER BY` inside `OVER` — defining row order within each window
- Ranking functions — `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`, and how they handle ties
- Offset functions — `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE` — looking at neighbors
- Window frames — `ROWS` vs `RANGE`, running totals, and moving averages
- Common pitfalls to carry forward

This notebook uses `customers` and `orders`. We add a few extra orders below so the window demos have meaningful data — multiple orders per customer, and at least one tie.

In [ ]:
%load_ext sql
%sql duckdb:///learn.db

In [ ]:
%%sql
-- Seed extra orders. Alice ends up with a 149.50 tie; Bob has 3 orders spaced over time.
DELETE FROM orders WHERE order_id BETWEEN 104 AND 107;
INSERT INTO orders VALUES
    (104, 1,  75.00, DATE '2024-05-02'),
    (105, 1, 149.50, DATE '2024-05-20'),
    (106, 2,  45.00, DATE '2024-05-01'),
    (107, 2, 200.00, DATE '2024-06-10');
SELECT * FROM orders ORDER BY customer_id, order_date;

## 1. The OVER clause — keep the row, compute over a window

A **window function** is an aggregate-like computation that runs over a set of rows (the *window*) and **returns one value per input row** — instead of collapsing the rows the way `GROUP BY` does.

Syntax: any aggregate or special function followed by `OVER (...)`. The `OVER` clause is what turns an ordinary function into a window function. Inside the parentheses you describe the window — which rows to include, in what order, and which subset of them this row should see.

An empty `OVER ()` means "the whole table is the window, every row sees all rows".

```
  GROUP BY                  OVER (PARTITION BY)
  +-----+-------+           +-----+-------+--------+
  | cid | total |           | cid | order | total  |  <- one row per *order*,
  +-----+-------+           +-----+-------+--------+     with the per-customer
  |  1  | 404.0 |           |  1  | 101   | 404.0  |     total attached.
  |  2  | 254.9 |           |  1  | 102   | 404.0  |
  +-----+-------+           |  1  | 104   | 404.0  |
   (rows collapsed)         |  1  | 105   | 404.0  |
                            |  2  | 103   | 254.9  |
                            |  ...| ...   | ...    |
                            +-----+-------+--------+
                             (rows preserved)
```

In [ ]:
%%sql
-- Empty OVER(): the whole orders table is one window.
-- Every row sees the same grand total and overall average.
SELECT order_id, amount,
       SUM(amount) OVER ()             AS grand_total,
       ROUND(AVG(amount) OVER (), 2)   AS overall_avg
FROM orders
ORDER BY order_id;

## 2. PARTITION BY — independent windows per group

`PARTITION BY` splits the rows into independent windows, one per distinct value of the partition expression. Each row only sees the rows in its own partition.

The shape of the result is identical to `GROUP BY` partition logic, but **no rows are collapsed** — every input row still appears, with the partition's aggregate attached. This is the workhorse pattern: each detail row alongside its own group total or average.

**Mental model:** `GROUP BY` collapses; `OVER (PARTITION BY)` preserves.

In [ ]:
%%sql
-- Each order alongside its customer's total and the order's share of that total.
SELECT order_id,
       customer_id,
       amount,
       SUM(amount)   OVER (PARTITION BY customer_id) AS customer_total,
       ROUND(100.0 * amount / SUM(amount) OVER (PARTITION BY customer_id), 1) AS pct_of_customer
FROM orders
ORDER BY customer_id, order_id;

## 3. ORDER BY inside OVER — order within the window

The `ORDER BY` *inside* an `OVER` clause is a completely different thing from the `ORDER BY` at the end of a query. The outer `ORDER BY` sorts the final result for display. The inner `ORDER BY` defines **the order rows appear within each window**, which is what lets the engine compute things like running totals and rank.

- For **ranking** and **offset** functions (`ROW_NUMBER`, `RANK`, `LAG`, `LEAD`), an `ORDER BY` inside `OVER` is **required** — the function has no meaning without a defined order.
- For **aggregate** window functions (`SUM`, `AVG`, etc.), adding an `ORDER BY` flips them from "see all rows in the partition" to "see only rows up to and including the current row" — i.e., **running** aggregates.

The two `ORDER BY`s are independent and often differ — for example, compute a running total in date order, then sort the output by customer name.

In [ ]:
%%sql
-- Running total per customer, ordered chronologically inside the window.
-- The outer ORDER BY just controls how the result is displayed.
SELECT customer_id,
       order_date,
       amount,
       SUM(amount) OVER (
           PARTITION BY customer_id
           ORDER BY order_date
       ) AS running_total
FROM orders
ORDER BY customer_id, order_date;

## 4. Ranking functions

Four ranking functions, all driven by the `OVER (PARTITION BY ... ORDER BY ...)` shape. They differ only in how they handle ties:

- **`ROW_NUMBER()`** — strict 1, 2, 3, ... — ties get broken arbitrarily.
- **`RANK()`** — 1, 2, 2, 4, ... — ties share a rank, then a gap.
- **`DENSE_RANK()`** — 1, 2, 2, 3, ... — ties share a rank, no gap.
- **`NTILE(n)`** — splits the partition into `n` roughly-equal buckets (1..n).

The classic use case is "top N per group": rank within partition, then keep ranks 1..N. Modern dialects support **`QUALIFY`** for filtering on window function results without wrapping in a subquery — Snowflake, BigQuery, DuckDB, and Databricks all support it; Postgres and SQL Server require the subquery form.

In [ ]:
%%sql
-- Same partition, same order, three different tie behaviors visible side-by-side.
-- Watch Alice's two 149.50 rows — they hit a tie.
SELECT customer_id, order_id, amount,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS row_num,
       RANK()       OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rnk,
       DENSE_RANK() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS dense_rnk
FROM orders
ORDER BY customer_id, amount DESC;

In [ ]:
%%sql
-- Top order per customer, using QUALIFY (DuckDB-supported syntax).
-- The portable Postgres/SQL Server equivalent wraps this in a CTE or subquery.
SELECT customer_id, order_id, amount
FROM orders
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) = 1;

## 5. Offset functions — looking at neighbors

Offset functions let a row see other rows in its own window:

- **`LAG(col, n)`** — the value of `col` from `n` rows earlier in the window. Default `n = 1`.
- **`LEAD(col, n)`** — the value of `col` from `n` rows later in the window.
- **`FIRST_VALUE(col)`** — the value of `col` in the first row of the window frame.
- **`LAST_VALUE(col)`** — the value of `col` in the **last row of the frame** (not the partition — see the frame trap in §7).

Both `LAG` and `LEAD` accept an optional default value for when the offset falls outside the partition — `LAG(amount, 1, 0)` returns 0 for the first row in each partition instead of `NULL`.

Classic use: month-over-month deltas, day-over-day deltas, gap-detection (compare each event's timestamp to the previous one in the same partition).

In [ ]:
%%sql
-- Each order alongside the customer's previous order amount and the delta.
-- The first order in each customer's history has NULL prev_amount (use a default to avoid).
SELECT customer_id,
       order_date,
       amount,
       LAG(amount)  OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_amount,
       amount - LAG(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS delta
FROM orders
ORDER BY customer_id, order_date;

## 6. Window frames — `ROWS` vs `RANGE`

The **frame** is the subset of the partition the function actually sees for the current row. By default:

- **Without `ORDER BY` inside `OVER`** — the frame is the whole partition.
- **With `ORDER BY` and no explicit frame** — the default frame is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. That gives you running aggregates, but the `RANGE` default has a sharp edge with ties (see §7).

Explicit frames have two flavors:

- **`ROWS BETWEEN ... AND ...`** — physical row counts. `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` is exactly three rows.
- **`RANGE BETWEEN ... AND ...`** — value ranges based on the `ORDER BY` column. All rows with the same value as the current row are treated as one position.

**Moving averages and trailing windows almost always want `ROWS`**, not `RANGE`. `RANGE` is for things like "all rows in the last 7 days" where the data isn't evenly spaced.

In [ ]:
%%sql
-- 3-row trailing average: this order plus the two before it (same customer).
-- Explicit ROWS frame; partitions with fewer than 3 rows average what they have.
SELECT customer_id,
       order_date,
       amount,
       ROUND(AVG(amount) OVER (
           PARTITION BY customer_id
           ORDER BY order_date
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ), 2) AS moving_avg_3
FROM orders
ORDER BY customer_id, order_date;

## 7. Common pitfalls (carry forward)

- **Default-frame surprise.** With `ORDER BY` inside `OVER` but no explicit frame, the frame defaults to `RANGE` (value-based, ties merged), not `ROWS` (row-count-based). For most running aggregates you actually want `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Be explicit.
- **`LAST_VALUE` returns the *current row*, not the partition's last row.** The default frame ends at `CURRENT ROW`, so `LAST_VALUE` of that frame is always the row you're on. To get the actual last value in the partition: `LAST_VALUE(col) OVER (... ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)`.
- **`WHERE` cannot reference window functions** — window functions run **after** `WHERE` (in fact, after `GROUP BY`/`HAVING`). To filter on a window result, wrap the query in a CTE or subquery, or use `QUALIFY` if your dialect supports it.
- **The two `ORDER BY`s are independent.** The one inside `OVER` controls window semantics; the one at the end controls how the final result is displayed. Don't assume one substitutes for the other.
- **`PARTITION BY` a high-cardinality column** creates many small windows and is expensive. If the partition column is unique per row, every window is one row and the function is doing useless work — that's usually a sign the logic is wrong.
- **`NTILE` does not handle ties specially.** Two rows with identical sort values can land in different buckets. Use `NTILE` only when bucket-boundary precision doesn't matter.
- **Window functions can't appear inside aggregate function arguments**, and aggregates can't appear inside `OVER (PARTITION BY ...)` expressions directly. When you need to combine them, use a CTE: aggregate first, then window over the result.

Next up: **notebook 08 — DDL, DML, Transactions & Query Planning**, the final notebook, where we shift from reading data to **changing** it — schema, inserts/updates/deletes, transactions, and a first look at `EXPLAIN`.